Business Context:

The bank is experiencing a higher-than-desired attrition rate in its premium "Gold" checking account segment. The marketing team wants to proactively identify customers at high risk of churning (closing all their accounts) in the next 90 days to target them with retention campaigns.

Data Schema
1. customers: customer_id, age, income_bracket, is_active (Boolean)
2. accounts: account_id, customer_id, account_type, status (e.g., 'Active', 'Closed'), open_date, close_date
3. transactions: transaction_id, account_id, transaction_date, amount, transaction_type
4. interactions: interaction_id, customer_id, interaction_date, interaction_type (e.g., 'Complaint, 'Service Inquiry')
Tasks:
Part A: SQL & Feature Engineering
1. Define your target variable has churned (1/0). A customer has "churned" if they have closed all their accounts within a specific historical observation window. (You will be given a specific cutoff date, e.g., "Consider accounts closed in the last 6 months as churners").
2. Write an SQL query to create a feature-rich dataset for model training. For each customer at a snapshot date (e.g., 90 days before churn/non-churn event), extract features like:
o num_accounts
o avg account_balance
o transaction_frequency_30d
o avg transaction_amount
o number_complaints_90d
o customer_tenure_days
o has_gold_account (Boolean)
Part B: Python Modeling
1. Using Python (Pandas, Scikit-learn), load the dataset you generated.
2. Perform necessary data preprocessing: handle missing values, encode categorical variables.
3. Split the data into training and test sets.
4. Train at least two classification models (e.g., Logistic Regression and Random Forest) to predict has churned.
5. Evaluate the models on the test set using appropriate metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC). Justify your choice of the primary metric for the business context.
Part C: Presentation
1. Present your findings to a non-technical business audience.
2. Explain which model you would deploy and why.
3. What are the top 3 features driving churn according to your best model? Explain what this tells the business about why customers might be leaving.
4. How would the marketing team use your model's output? What are the potential risks or ethical considerations of using such a model?


In [ ]:
# Import labraries
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve

: 

In [ ]:
# --- 1. Load Datasets
account = pd.read_csv(r"C:\Users\HP\Desktop\Union_B_project\datasets\accounts.csv")
customer = pd.read_csv(r"C:\Users\HP\Desktop\Union_B_project\datasets\customers.csv")
interaction = pd.read_csv(r"C:\Users\HP\Desktop\Union_B_project\datasets\interactions.csv")
transaction = pd.read_csv(r"C:\Users\HP\Desktop\Union_B_project\datasets\transactions.csv")


In [ ]:
customer.head()

In [ ]:
account.head()

In [ ]:
transaction.head()

In [ ]:
interaction.head()

In [ ]:
# --- 2. Data Type Conversion and Date Setup ---
SNAPSHOT_DATE = datetime(2025, 7, 30)
PREDICTION_END_DATE = SNAPSHOT_DATE + timedelta(days=90)

In [ ]:
# Convert all date columns to datetime objects
customer['cust_creation_date'] = pd.to_datetime(customer['cust_creation_date'], format='%d/%m/%Y', errors='coerce')
account['open_date'] = pd.to_datetime(account['open_date'])
account['close_date'] = pd.to_datetime(account['close_date'])
transaction['transaction_date'] = pd.to_datetime(transaction['transaction_date'])
interaction['interaction_date'] = pd.to_datetime(interaction['interaction_date'])

In [ ]:
# --- 3. Define the Target Variable: `has_churned` ---

# Step 3a: Identify customers with ACTIVE accounts on the Snapshot Date (T)
active_on_T = account[
    (account['open_date'] <= SNAPSHOT_DATE) &
    (account['close_date'].isnull() | (account['close_date'] > SNAPSHOT_DATE))
].groupby('customer_id')['account_id'].nunique().reset_index()
active_on_T.columns = ['customer_id', 'active_accounts_on_T']
active_on_T = active_on_T[active_on_T['active_accounts_on_T'] > 0]
model_population = active_on_T[['customer_id']]

In [ ]:
# Step 3b: Determine if ALL accounts were closed in the Prediction Window (T+1 to T+90)
customer_account_status = account[account['open_date'] <= SNAPSHOT_DATE].copy()
customer_latest_close = customer_account_status.groupby('customer_id')['close_date'].max().reset_index()

In [ ]:
# Merge with the model population
churn_df = model_population.merge(customer_latest_close, on='customer_id', how='left')
churn_df['is_churner'] = (churn_df['close_date'] >= SNAPSHOT_DATE + timedelta(days=1)) & \
                         (churn_df['close_date'] <= PREDICTION_END_DATE) & \
                         (churn_df['close_date'].notnull())

In [ ]:
# Final Target Variable
churn_df['has_churned'] = churn_df['is_churner'].astype(int)
base_df = customer[['customer_id', 'cust_creation_date', 'age', 'income_bracket', 'region']]
base_df = base_df.merge(churn_df[['customer_id', 'has_churned']], on='customer_id', how='left')

In [ ]:
# --- 4. Feature Engineering (as of Snapshot Date 2025-07-30) ---

# Feature 1: `customer_tenure_days`
base_df['customer_tenure_days'] = (SNAPSHOT_DATE - base_df['cust_creation_date']).dt.days



In [ ]:
# Feature 2: `has_gold_account`
gold_accounts_T = account[
    (account['account_type'] == 'Gold Checking') &
    (account['open_date'] <= SNAPSHOT_DATE) &
    (account['close_date'].isnull() | (account['close_date'] > SNAPSHOT_DATE))
]['customer_id'].unique()
base_df['has_gold_account'] = base_df['customer_id'].isin(gold_accounts_T).astype(int)

# --- Account-Level Features ---
active_accounts_T = account[
    (account['open_date'] <= SNAPSHOT_DATE) &
    (account['close_date'].isnull() | (account['close_date'] > SNAPSHOT_DATE))
].copy()

In [ ]:
# Feature 3: `num_accounts`
num_accounts_T = active_accounts_T.groupby('customer_id').size().reset_index(name='num_accounts')
base_df = base_df.merge(num_accounts_T, on='customer_id', how='left').fillna({'num_accounts': 0})

# --- Transaction-Level Features ---
transactions_T = transaction[transaction['transaction_date'] <= SNAPSHOT_DATE].copy()
TXN_WINDOW = 180
txn_T_window = transactions_T[transactions_T['transaction_date'] > SNAPSHOT_DATE - timedelta(days=TXN_WINDOW)]
txn_T_window = txn_T_window.merge(active_accounts_T[['account_id', 'customer_id']], on='account_id', how='inner')

In [ ]:
# Feature 4: `avg_net_transaction_amount_180d` (Approximation for 'avg account_balance')
txn_T_window['signed_amount'] = np.where(txn_T_window['transaction_type'] == 'Credit', txn_T_window['amount'],
                                        np.where(txn_T_window['transaction_type'] == 'Debit', -txn_T_window['amount'],
                                                 0))
net_txn_180d = txn_T_window.groupby('customer_id')['signed_amount'].mean().reset_index(name='avg_net_transaction_amount_180d')
base_df = base_df.merge(net_txn_180d, on='customer_id', how='left')

In [ ]:
# Feature 5: `transaction_frequency_30d`
TXN_FREQ_WINDOW = 30
txn_T_30d = transactions_T[transactions_T['transaction_date'] > SNAPSHOT_DATE - timedelta(days=TXN_FREQ_WINDOW)]
txn_T_30d = txn_T_30d.merge(active_accounts_T[['account_id', 'customer_id']], on='account_id', how='inner')

txn_frequency = txn_T_30d.groupby('customer_id').size().reset_index(name='transaction_frequency_30d')
base_df = base_df.merge(txn_frequency, on='customer_id', how='left')

In [ ]:
# Feature 6: `avg_transaction_amount`
avg_txn_amount = txn_T_window.groupby('customer_id')['amount'].mean().reset_index(name='avg_transaction_amount')
base_df = base_df.merge(avg_txn_amount, on='customer_id', how='left')


# --- Interaction-Level Features ---
interactions_T = interaction[interaction['interaction_date'] <= SNAPSHOT_DATE].copy()

In [ ]:
# --- NEW Feature Engineering based on account_type (to capture product mix) ---

# 1. num_distinct_account_types (captures stickiness)
num_account_types = active_accounts_T.groupby('customer_id')['account_type'].nunique().reset_index(name='num_distinct_account_types')
base_df = base_df.merge(num_account_types, on='customer_id', how='left').fillna({'num_distinct_account_types': 0})

# 2. has_savings_account (a common indicator of primary bank relationship)
# Checking for 'Savings' or 'Money Market' which represent liquid, non-checking investments
key_deposit_types = ['Savings', 'Money Market']
has_savings_account = active_accounts_T[
    active_accounts_T['account_type'].isin(key_deposit_types)
]['customer_id'].unique()
base_df['has_savings_account'] = base_df['customer_id'].isin(has_savings_account).astype(int)

# 3. num_other_checking_accounts
num_other_checking_accounts = active_accounts_T[
    (active_accounts_T['account_type'] == 'Checking')
].groupby('customer_id').size().reset_index(name='num_other_checking_accounts')
base_df = base_df.merge(num_other_checking_accounts, on='customer_id', how='left').fillna({'num_other_checking_accounts': 0})


In [ ]:
# Feature 7: `number_complaints_90d`
interactions_T = interaction[interaction['interaction_date'] <= SNAPSHOT_DATE].copy()
COMPLAINT_WINDOW = 90
interactions_T_90d = interactions_T[interactions_T['interaction_date'] > SNAPSHOT_DATE - timedelta(days=COMPLAINT_WINDOW)]
complaints_90d = interactions_T_90d[interactions_T_90d['interaction_type'] == 'Complaint'] \
    .groupby('customer_id').size().reset_index(name='number_complaints_90d')
base_df = base_df.merge(complaints_90d, on='customer_id', how='left')



In [ ]:
# Final Feature Matrix Cleanup
feature_cols = [
    'customer_id', 'has_churned', 'age', 'income_bracket', 'region',
    'has_gold_account', 'customer_tenure_days', 'num_accounts',
    'avg_net_transaction_amount_180d', 'transaction_frequency_30d',
    'avg_transaction_amount', 'number_complaints_90d',
    'num_distinct_account_types',
    'has_savings_account',
    'num_other_checking_accounts'
]
final_df = base_df[feature_cols].copy()

In [ ]:
display(final_df.head())

In [ ]:
final_df.info()

In [ ]:
# --- Data Preparation for EDA ---
df = final_df.copy()
df['has_churned'] = df['has_churned'].fillna(0).astype(int)

# Fill 0 for all aggregated features (NaN means zero activity/count)
fill_zero_cols = [
    'avg_net_transaction_amount_180d', 'transaction_frequency_30d',
    'avg_transaction_amount', 'number_complaints_90d',
    'num_distinct_account_types', 'has_savings_account',
    'num_other_checking_accounts', 'num_accounts'
]
df[fill_zero_cols] = df[fill_zero_cols].fillna(0)
df.to_csv('cleaned_churn_features.csv', index=False)

In [ ]:
df.head()

In [ ]:
# --- EDA Plot Generation ---
plot_features = [
    ('income_bracket', 'bar', 'Income Bracket vs. Churn Rate'),
    ('number_complaints_90d', 'bar', 'Number of Complaints vs. Churn Rate'), # Changed to bar plot
    ('customer_tenure_days', 'kde', 'Customer Tenure Distribution'),
    ('has_savings_account', 'bar', 'Has Savings Account vs. Churn Rate')
]

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Exploratory Data Analysis: Key Drivers of Churn', fontsize=18, y=1.02)
axes = axes.flatten()

for i, (col, plot_type, title) in enumerate(plot_features):
    ax = axes[i]

    if plot_type == 'bar':
        churn_rate = df.groupby(col)['has_churned'].mean().reset_index(name='Churn Rate')

        order = None
        if col == 'income_bracket':
            order = ['Low', 'Medium', 'High']
        elif col == 'has_savings_account':
            order = [0, 1]
        # For 'number_complaints_90d', we can let seaborn sort numerically or manually specify if preferred

        sns.barplot(x=col, y='Churn Rate', data=churn_rate, order=order, ax=ax, palette='viridis')
        ax.set_title(title)
        ax.set_ylabel('Churn Rate (%)')
        ax.set_xlabel(col.replace('_', ' ').title())

    elif plot_type == 'kde':
        df_plot = df[df[col] < df[col].quantile(0.99)] # Filter extreme outliers for better visualization

        sns.kdeplot(data=df_plot, x=col, hue='has_churned', fill=True, common_norm=False,
                    palette={0: 'blue', 1: 'red'}, alpha=0.5, ax=ax, legend=True)
        ax.set_title(title)
        ax.set_ylabel('Density')
        ax.set_xlabel(col.replace('_', ' ').title())
        ax.legend(title='Churned', labels=['No (0)', 'Yes (1)'])

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('eda_churn_subplots.png')
# Output: 'eda_churn_subplots.png'


In [ ]:
# Calculate key insights
overall_churn_rate = df['has_churned'].mean() * 100
average_tenure = df['customer_tenure_days'].mean()
average_txn_frequency = df['transaction_frequency_30d'].mean()
gold_account_holders = df['has_gold_account'].mean() * 100
customers_with_complaints_count = (df['number_complaints_90d'] > 0).sum()
customers_with_complaints_percentage = (customers_with_complaints_count / len(df)) * 100

print(f"Overall churn rate: {overall_churn_rate:.2f}%")
print(f"Average customer tenure: {average_tenure:.0f} days")
print(f"Average transaction frequency: {average_txn_frequency:.1f} transactions/30d")
print(f"Gold account holders: {gold_account_holders:.1f}%")
print(f"Customers with complaints: {customers_with_complaints_count} ({customers_with_complaints_percentage:.1f}%)")

### **Key Insights & Actionable Recommendations**

**KEY INSIGHTS**

1.  Overall churn rate: **1.50%**
2.  Average customer tenure: **1253 days**
3.  Average transaction frequency: **5.7 transactions/30d**
4.  Gold account holders: **5.2%**
5.  Customers with complaints: **213 (10.7%)**

**ACTIONABLE INSIGHTS:**
-------------------
*   Monitor age group with highest churn rate (refer to EDA)
*   Focus retention efforts on income brackets with high churn (refer to EDA)
*   Investigate regional differences in customer behavior (refer to EDA)
*   Analyze correlation between complaints and churn; prioritize resolving complaints swiftly, especially given its importance as a churn driver.
*   Identify opportunities for Gold account upselling and cross-selling additional products, as multi-product customers (e.g., those with savings accounts or more total accounts) tend to be more loyal.

# DATA PREPARATION AND MODELLING

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [ ]:
# Identify features (X) and target (y)
X = df.drop(['customer_id', 'has_churned'], axis=1)
y = df['has_churned']

In [ ]:
# Identify column types
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

# Define Preprocessing Pipelines
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_features),
    ('categorical', categorical_pipeline, categorical_features)
])

In [ ]:
# --- 2. Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# Note: Corrected the order of y_train and y_test assignment

In [ ]:
from imblearn.over_sampling import SMOTE

# --- Preprocess the training and test data BEFORE applying SMOTE ---
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Apply SMOTE to the PROCESSED training data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_processed, y_train)

# --- 3. Train Classification Models ---
models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', random_state=42, class_weight='balanced', max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', max_depth=10, min_samples_leaf=5)
}

trained_models = {}
results = {}

for name, model in models.items():
    classifier_pipeline = Pipeline([
        ('classifier', model)
    ])

    classifier_pipeline.fit(X_resampled, y_resampled)
    trained_models[name] = classifier_pipeline

    y_pred = classifier_pipeline.predict(X_test_processed)
    y_pred_proba = classifier_pipeline.predict_proba(X_test_processed)[:, 1]

    # Evaluate
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    results[name] = {
        'Accuracy': (y_pred == y_test).mean(),
        'Precision (Churn)': report['1']['precision'],
        'Recall (Churn)': report['1']['recall'],
        'F1-Score (Churn)': report['1']['f1-score'],
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    print(f"Model: {name}")
    print(f"Accuracy: {(y_pred == y_test).mean()}")
    print(f"Precision (Churn): {report['1']['precision']}")
    print(f"Recall (Churn): {report['1']['recall']}")
    print(f"F1-Score (Churn): {report['1']['f1-score']}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba)}")
    print("\n")

### Threshold Tuning for Random Forest Model

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score, auc

# Get predicted probabilities for the positive class (churn = 1) from the Random Forest model
rf_model_results = results['Random Forest']
y_pred_proba_rf = rf_model_results['y_pred_proba']

# Calculate precision, recall, and thresholds
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba_rf)

# Initialize lists to store metrics for various thresholds
thresholds_tuned = []
precisions_tuned = []
recalls_tuned = []
f1_scores_tuned = []

# Evaluate metrics for a range of thresholds
# We exclude the last threshold from precision_recall_curve as it's typically 1 and recall is 0
for t in thresholds:
    if t == 1.0: continue # Skip the threshold 1.0 where precision/recall might be undefined
    y_pred_tuned = (y_pred_proba_rf >= t).astype(int)

    # Calculate metrics for the positive class (1)
    report_tuned = classification_report(y_test, y_pred_tuned, output_dict=True, zero_division=0)

    if '1' in report_tuned:
        precisions_tuned.append(report_tuned['1']['precision'])
        recalls_tuned.append(report_tuned['1']['recall'])
        f1_scores_tuned.append(report_tuned['1']['f1-score'])
        thresholds_tuned.append(t)
    else:
        # If no positive predictions, set metrics to 0
        precisions_tuned.append(0.0)
        recalls_tuned.append(0.0)
        f1_scores_tuned.append(0.0)
        thresholds_tuned.append(t)

# Create a DataFrame for easy plotting
threshold_df = pd.DataFrame({
    'Threshold': thresholds_tuned,
    'Precision': precisions_tuned,
    'Recall': recalls_tuned,
    'F1-Score': f1_scores_tuned
})

# Plotting Precision, Recall, and F1-Score vs. Threshold
plt.figure(figsize=(12, 7))
plt.plot(threshold_df['Threshold'], threshold_df['Precision'], label='Precision (Churn)')
plt.plot(threshold_df['Threshold'], threshold_df['Recall'], label='Recall (Churn)')
plt.plot(threshold_df['Threshold'], threshold_df['F1-Score'], label='F1-Score (Churn)')

plt.xlabel('Probability Threshold')
plt.ylabel('Score')
plt.title('Random Forest: Precision, Recall, and F1-Score vs. Threshold')
plt.legend()
plt.grid(True)
plt.xlim(0, 1) # Set x-axis limits to 0 to 1 for probability thresholds
plt.ylim(0, 1) # Set y-axis limits to 0 to 1 for scores
plt.axvline(x=0.5, color='r', linestyle='--', label='Default Threshold (0.5)') # Indicate default threshold
plt.tight_layout()
plt.show()

# Find the threshold that maximizes F1-Score (a common balance between P and R)
optimal_threshold_f1 = threshold_df.loc[threshold_df['F1-Score'].idxmax()]
print(f"\nOptimal Threshold (maximizing F1-Score): {optimal_threshold_f1['Threshold']:.4f}")
print(f"Precision at this threshold: {optimal_threshold_f1['Precision']:.4f}")
print(f"Recall at this threshold: {optimal_threshold_f1['Recall']:.4f}")
print(f"F1-Score at this threshold: {optimal_threshold_f1['F1-Score']:.4f}")

### Apply SMOTE for Class Imbalance

In [ ]:
# !pip install imblearn

from imblearn.over_sampling import SMOTE

# --- Preprocess the training and test data BEFORE applying SMOTE ---
# Fit and transform X_train using the preprocessor
X_train_processed = preprocessor.fit_transform(X_train)
# Transform X_test using the *fitted* preprocessor
X_test_processed = preprocessor.transform(X_test)

print("Original training set class distribution:")
print(y_train.value_counts())

# Apply SMOTE to the PROCESSED training data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_processed, y_train)

print("\nResampled training set class distribution:")
print(y_resampled.value_counts())

Now that the training data is balanced using SMOTE, I will modify the model training and evaluation cell to use this `X_resampled` and `y_resampled` data. Please run the SMOTE cell above first, then the modified model training cell, and finally the plotting cell to see the updated results.

In [ ]:
# --- 4. Plotting and Visualization (3x2 Subplots) ---

fig, axes = plt.subplots(3, 2, figsize=(18, 20))
fig.suptitle('Model Evaluation Results (Test Set)', fontsize=20, y=1.02)
axes = axes.flatten()

# Get optimal threshold for Random Forest (assuming it was calculated in 'd380e1d5')
# If optimal_threshold_f1 variable is not available, default to 0.5
optimal_rf_threshold = 0.5
if 'optimal_threshold_f1' in locals():
    optimal_rf_threshold = optimal_threshold_f1['Threshold']


# 4a. Confusion Matrix Heatmaps
labels = ['Did Not Churn (0)', 'Churned (1)']
for i, (name, res) in enumerate(results.items()):
    ax = axes[i]

    # Use the optimal threshold for Random Forest's confusion matrix
    current_y_pred = res['y_pred']
    if name == 'Random Forest':
        current_y_pred = (res['y_pred_proba'] >= optimal_rf_threshold).astype(int)

    cm = confusion_matrix(y_test, current_y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=labels, yticklabels=labels)
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

# 4b. Model Performance Comparison Bar Chart
ax = axes[2]
metrics_to_plot = ['Recall (Churn)', 'Precision (Churn)', 'F1-Score (Churn)', 'ROC-AUC'] # Added F1-Score (Churn)
results_df = pd.DataFrame({k: {m: results[k][m] for m in metrics_to_plot} for k in results}).T
results_df.index.name = 'Model'

# Update Random Forest metrics with optimal threshold for the bar chart as well
if 'Random Forest' in results_df.index:
    y_pred_rf_tuned = (results['Random Forest']['y_pred_proba'] >= optimal_rf_threshold).astype(int)
    report_rf_tuned = classification_report(y_test, y_pred_rf_tuned, output_dict=True, zero_division=0)
    results_df.loc['Random Forest', 'Precision (Churn)'] = report_rf_tuned['1']['precision']
    results_df.loc['Random Forest', 'Recall (Churn)'] = report_rf_tuned['1']['recall']
    results_df.loc['Random Forest', 'F1-Score (Churn)'] = report_rf_tuned['1']['f1-score']
    # Accuracy and ROC-AUC don't change with threshold tuning on y_pred, only y_pred_proba

results_df.plot(kind='bar', rot=0, colormap='viridis', ax=ax)
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.0)
ax.legend(title='Metric')
ax.grid(axis='y', linestyle='--')

# 4c. ROC Curve
ax = axes[3]
ax.set_title('Receiver Operating Characteristic (ROC) Curve')
ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.50)') # Diagonal line (random guessing)

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_pred_proba'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')

# Add a vertical line for the optimal threshold on the Random Forest ROC curve (if applicable)
if 'Random Forest' in results and 'optimal_threshold_f1' in locals():
    # Find the corresponding FPR and TPR for the optimal threshold
    y_pred_proba_rf = results['Random Forest']['y_pred_proba']
    precision_rf_tuned, recall_rf_tuned, thresholds_rf = precision_recall_curve(y_test, y_pred_proba_rf)
    # Find index of threshold closest to optimal_rf_threshold
    idx = np.argmin(np.abs(thresholds_rf - optimal_rf_threshold))
    # Get corresponding recall (TPR) and precision (not directly plotted on ROC but for context)
    tpr_at_optimal_threshold = recall_rf_tuned[idx]

    pass # Keeping it clean, as direct threshold marking on ROC is less common than PR curve

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.legend(loc='lower right')
ax.grid(True)

# 4d. Top 10 Feature Importances (Random Forest)
ax = axes[4]
rf_pipeline = trained_models['Random Forest']
# The preprocessor is no longer part of the pipeline here because the data is already processed in X_resampled
# We need to recreate the preprocessor for feature name extraction
ohe_feature_names = preprocessor.transformers_[1][1].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + ohe_feature_names.tolist()
importances = rf_pipeline['classifier'].feature_importances_
top_10_features = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(10)

top_10_features.sort_values().plot(kind='barh', color='darkred', ax=ax)
ax.set_title('Top 10 Feature Importances (Random Forest)')
ax.set_xlabel('Feature Importance Score')
ax.set_ylabel('Feature')

# 4e. Model Accuracy Comparison
ax = axes[5]
accuracy_data = {
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [results['Logistic Regression']['Accuracy'], results['Random Forest']['Accuracy']]
}
accuracy_df = pd.DataFrame(accuracy_data)
sns.barplot(x='Model', y='Accuracy', data=accuracy_df, hue='Model', palette='magma', ax=ax, legend=False)
ax.set_title('Model Accuracy Comparison')
ax.set_ylabel('Accuracy Score')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('model_evaluation_subplots.png')
plt.show()
print("Saved model_evaluation_subplots.png")

### Visualizing Random Forest Performance for Stakeholders

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

# Prepare data for the summary table
summary_data = {}
for name, res in results.items():
    summary_data[name] = {
        'Accuracy': res['Accuracy'],
        'Precision (Churn)': res['Precision (Churn)'],
        'Recall (Churn)': res['Recall (Churn)'],
        'F1-Score (Churn)': res['F1-Score (Churn)'],
        'ROC-AUC': res['ROC-AUC']
    }

# --- Apply optimal threshold for Random Forest if available and update summary_data ---
# Ensure optimal_threshold_f1 and y_test (and y_pred_proba for RF) are available in the kernel
if 'optimal_threshold_f1' in globals() and 'Random Forest' in results and 'y_test' in globals():
    optimal_rf_threshold = optimal_threshold_f1['Threshold']
    y_pred_proba_rf_tuned = results['Random Forest']['y_pred_proba']
    y_pred_rf_tuned = (y_pred_proba_rf_tuned >= optimal_rf_threshold).astype(int)

    # Recalculate metrics for Random Forest using the tuned threshold
    report_rf_tuned = classification_report(y_test, y_pred_rf_tuned, output_dict=True, zero_division=0)

    # Update summary_data for Random Forest with tuned metrics
    summary_data['Random Forest']['Accuracy'] = (y_pred_rf_tuned == y_test).mean()
    summary_data['Random Forest']['Precision (Churn)'] = report_rf_tuned['1']['precision']
    summary_data['Random Forest']['Recall (Churn)'] = report_rf_tuned['1']['recall']
    summary_data['Random Forest']['F1-Score (Churn)'] = report_rf_tuned['1']['f1-score']
    # ROC-AUC remains the same as it's threshold-independent


# Create a DataFrame from the summary data
model_summary_df = pd.DataFrame(summary_data).T
model_summary_df.index.name = 'Model'

# Display the formatted table
print("\n--- Comprehensive Model Performance Summary ---")
display(model_summary_df.round(4))

### Random Forest Performance at Optimal Threshold (Bar Chart)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure optimal_threshold_f1 is available from previous execution
if 'optimal_threshold_f1' not in globals():
    print("Please run the 'Threshold Tuning for Random Forest Model' cell first.")
else:
    metrics_at_optimal_threshold = pd.DataFrame({
        'Metric': ['Precision (Churn)', 'Recall (Churn)'],
        'Score': [optimal_threshold_f1['Precision'], optimal_threshold_f1['Recall']]
    })

    plt.figure(figsize=(8, 6))
    sns.barplot(x='Metric', y='Score', data=metrics_at_optimal_threshold, palette=['#1f77b4', '#ff7f0e'])
    plt.title(f'Random Forest Performance at Optimal F1-Score Threshold ({optimal_threshold_f1["Threshold"]:.4f})')
    plt.ylim(0, 0.4) # Adjust y-limit to better show differences (max recall is ~0.33)
    plt.ylabel('Score')
    plt.xlabel('')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

### Model Performance Summary Table

In [ ]:
from sklearn.metrics import classification_report

# Ensure optimal_threshold_f1 and y_pred_proba_rf are available
if 'optimal_threshold_f1' in globals() and 'y_pred_proba_rf' in globals():
    optimal_rf_threshold = optimal_threshold_f1['Threshold']

    # Generate predictions using the optimal threshold
    y_pred_tuned_rf = (y_pred_proba_rf >= optimal_rf_threshold).astype(int)

    # Print the classification report
    print(f"Classification Report for Random Forest Model (Optimal Threshold: {optimal_rf_threshold:.4f}):\n")
    print(classification_report(y_test, y_pred_tuned_rf, zero_division=0))
else:
    print("Optimal threshold or predicted probabilities for Random Forest are not available.")
    print("Please ensure the 'Threshold Tuning for Random Forest Model' cell has been executed.")

### Save Final Model and Artifacts

To ensure our trained models and preprocessor can be reused for future predictions or deployment, we'll save them using `joblib`.

In [ ]:
import joblib
import os

# Define a directory to save the models
model_dir = 'churn_models'
os.makedirs(model_dir, exist_ok=True)

# --- Save the Preprocessor ---
preprocessor_path = os.path.join(model_dir, 'preprocessor.joblib')
joblib.dump(preprocessor, preprocessor_path)
print(f"Preprocessor saved to: {preprocessor_path}")

# --- Save each trained model ---
for name, model_pipeline in trained_models.items():
    model_path = os.path.join(model_dir, f'{name.lower().replace(" ", "_")}_model.joblib')
    joblib.dump(model_pipeline, model_path)
    print(f"Model '{name}' saved to: {model_path}")

### Loading the Saved Models and Preprocessor

To demonstrate, here's how you would load the saved preprocessor and one of the models (e.g., Random Forest) for making new predictions:

In [ ]:
import joblib
import os

# Define the directory where models are saved
model_dir = 'churn_models'

# --- Load the Preprocessor ---
loaded_preprocessor = joblib.load(os.path.join(model_dir, 'preprocessor.joblib'))
print(f"Preprocessor loaded from: {os.path.join(model_dir, 'preprocessor.joblib')}")

# --- Load a specific model (e.g., Random Forest) ---
loaded_rf_model = joblib.load(os.path.join(model_dir, 'random_forest_model.joblib'))
print(f"Random Forest Model loaded from: {os.path.join(model_dir, 'random_forest_model.joblib')}")

# Now you can use these to make predictions on new data:
# Example (assuming you have 'new_customer_data' as a pandas DataFrame):
# new_customer_data_processed = loaded_preprocessor.transform(new_customer_data)
# predictions = loaded_rf_model.predict(new_customer_data_processed)
# probabilities = loaded_rf_model.predict_proba(new_customer_data_processed)[:, 1]